## CE2 Phase 1: Train-Test Split

### Motivation: Generalization vs. Memorization
In statistical machine learning, the fundamental objective is **generalization**—learning systematic patterns from training observations that reliably transfer to unseen data.
- If a regression model is evaluated on the exact same data on which it was trained, the resulting error metrics will be unrealistically optimistic. The model can simply memorize sample-specific noise, variance, and idiosyncratic outliers rather than genuine predictive relationships.
- To obtain an honest, unbiased assessment of predictive performance, we partition the dataset to hold out a dedicated **test set** before model training. This test set remains strictly untouched throughout fitting and hyperparameter selection, acting as a realistic proxy for unseen real-world data.

### Partitioning Strategy: 80/20 Ratio with `random_state=42`
- **80/20 Split Proportion:** With our dataset of $N = 2,998$ observations, allocating $80\%$ to training ($N_{\text{train}} = 2,398$) provides sufficient statistical power to estimate parameters across 14 predictor features. The remaining $20\%$ holdout ($N_{\text{test}} = 600$) is large enough to ensure evaluation metrics (MAE, RMSE, $R^2$) are statistically stable and representative.
- **Deterministic Seed (`random_state=42`):** Setting a fixed random seed ensures pseudo-random shuffling produces identical, reproducible partitions across different runs, environments, and evaluator machines.


In [1]:
# Ensure working directory is set to project root
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Imports for data manipulation, splitting, and scaling
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load feature matrix X and target vector y from CE1 processed files
X = pd.read_csv("data/processed/X_features.csv")
y = pd.read_csv("data/processed/y_target.csv")["age"]

# Print dataset dimensions and verify row alignment
print("=== BASELINE DATASET LOADED ===")
print(f"X shape (rows, features): {X.shape}")
print(f"y shape (target instances): {y.shape}")
print(f"Row count alignment confirmed (len(X) == len(y)): {len(X) == len(y)}")


=== BASELINE DATASET LOADED ===
X shape (rows, features): (2998, 14)
y shape (target instances): (2998,)
Row count alignment confirmed (len(X) == len(y)): True


In [2]:
# Partition dataset into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Compute percentage proportions
train_pct = (len(X_train) / len(X)) * 100
test_pct = (len(X_test) / len(X)) * 100

# Print partition shapes and percentage breakdown
print("=== TRAIN-TEST SPLIT PARTITION SUMMARY ===")
print(f"X_train shape: {X_train.shape} ({train_pct:.1f}% of dataset)")
print(f"X_test shape:  {X_test.shape} ({test_pct:.1f}% of dataset)")
print(f"y_train shape: {y_train.shape} ({train_pct:.1f}% of dataset)")
print(f"y_test shape:  {y_test.shape} ({test_pct:.1f}% of dataset)")


=== TRAIN-TEST SPLIT PARTITION SUMMARY ===
X_train shape: (2398, 14) (80.0% of dataset)
X_test shape:  (600, 14) (20.0% of dataset)
y_train shape: (2398,) (80.0% of dataset)
y_test shape:  (600,) (20.0% of dataset)


In [3]:
# Sanity-check: Verify that target variable (age) distribution is balanced between train and test splits
train_desc = y_train.describe()
test_desc = y_test.describe()

# Display descriptive statistics side-by-side
split_comparison = pd.DataFrame({
    f"y_train (n={len(y_train)})": train_desc,
    f"y_test (n={len(y_test)})": test_desc
})

print("=== TARGET VARIABLE DISTRIBUTION COMPARISON (TRAIN vs. TEST) ===")
print(split_comparison.round(2))
print("\nObservation: Mean, standard deviation, quartiles, and range are closely aligned between train and test splits, confirming representative partitioning.")


=== TARGET VARIABLE DISTRIBUTION COMPARISON (TRAIN vs. TEST) ===
       y_train (n=2398)  y_test (n=600)
count           2398.00          600.00
mean              33.22           34.18
std               20.06           21.10
min                1.00            1.00
25%               23.00           22.00
50%               29.00           30.00
75%               45.00           47.00
max              110.00          100.00

Observation: Mean, standard deviation, quartiles, and range are closely aligned between train and test splits, confirming representative partitioning.


## CE2 Phase 2: Feature Scaling

### Why Scales Differ Across Features
The feature matrix $X$ contains 14 predictor variables operating on three distinctly different numerical scales:
1. **Raw Facial Landmark Measurements (Pixels):** `eye_distance`, `mouth_width`, `face_width`, and `face_height` range between approximately $\sim 50$ and $\sim 220$ pixels.
2. **Engineered Proportional Ratios:** `face_aspect_ratio`, `eye_to_face_ratio`, and `mouth_to_face_ratio` are scale-invariant dimensionless ratios ranging roughly between $\sim 0.2$ and $\sim 1.5$.
3. **One-Hot Encoded Demographics:** `gender_0`, `gender_1`, and `race_0` through `race_4` are binary indicator flags strictly taking values in $\{0, 1\}$.

Without feature scaling, optimization algorithms and regularized regression penalties would be dominated by features with arbitrarily larger numerical ranges (such as pixel widths) rather than features with genuine predictive association.

### Standardization Formula (StandardScaler)
We use z-score standardization:
$$z = \frac{x - \mu}{\sigma}$$
where $\mu$ is the empirical feature mean and $\sigma$ is the empirical standard deviation. After transformation, each continuous feature has a mean of approximately $0$ and a standard deviation (variance) of $1$.

### Why Fit on Training Data ONLY (Preventing Data Leakage)
- **Data Leakage Risk:** In a valid evaluation framework, the test set must represent unseen future instances. If the scaler computed its parameters (mean $\mu$ and standard deviation $\sigma$) across the entire dataset or using test observations, information about the test distribution would contaminate the training pipeline.
- **Protocol:** We call `.fit()` **strictly on `X_train`** to learn $\mu_{\text{train}}$ and $\sigma_{\text{train}}$. We then use those exact training parameters to `.transform()` both `X_train` and `X_test`.

### Why One-Hot Encoded Columns Are NOT Scaled
One-hot encoded columns are binary indicators ($\{0, 1\}$) representing category membership. Standardizing them destroys their natural discrete interpretation, converts intuitive zero-values into non-zero continuous floats, and provides no mathematical benefit since their values are already bounded within $[0, 1]$.


In [4]:
# Define the 7 continuous numerical features requiring standardization
continuous_cols = [
    'eye_distance', 'mouth_width', 'face_width', 'face_height',
    'face_aspect_ratio', 'eye_to_face_ratio', 'mouth_to_face_ratio'
]

# Create explicit copies to preserve unscaled original DataFrames
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Initialize StandardScaler and fit STRICTLY on training continuous columns
scaler = StandardScaler()
scaler.fit(X_train[continuous_cols])

# Transform continuous columns on both train and test sets using training parameters
X_train_scaled[continuous_cols] = scaler.transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])
# Note: All one-hot encoded columns (gender_0, gender_1, race_0-race_4) remain unscaled

# Display learned scaler parameters (mean and standard deviation)
scaler_params_df = pd.DataFrame({
    "Learned Mean (mu)": scaler.mean_,
    "Learned Scale (std sigma)": scaler.scale_
}, index=continuous_cols)

print("=== SCALER PARAMETERS (LEARNED FROM X_train ONLY) ===")
print(scaler_params_df.round(4))


=== SCALER PARAMETERS (LEARNED FROM X_train ONLY) ===
                     Learned Mean (mu)  Learned Scale (std sigma)
eye_distance                   78.0924                     5.7656
mouth_width                    65.1807                     6.8079
face_width                    168.0535                    11.2856
face_height                   194.2031                     7.5108
face_aspect_ratio               1.1601                     0.0804
eye_to_face_ratio               0.4651                     0.0245
mouth_to_face_ratio             0.3884                     0.0375


In [5]:
# Verification of scaled distributions
train_means = X_train_scaled[continuous_cols].mean()
train_stds = X_train_scaled[continuous_cols].std()

test_means = X_test_scaled[continuous_cols].mean()
test_stds = X_test_scaled[continuous_cols].std()

# Build verification summary tables
train_verif = pd.DataFrame({"Mean": train_means, "Std": train_stds})
test_verif = pd.DataFrame({"Mean": test_means, "Std": test_stds})

print("=== SCALING VERIFICATION: X_train_scaled (Expected: Mean ≈ 0.0, Std ≈ 1.0) ===")
print(train_verif.round(6))

print("\n=== SCALING VERIFICATION: X_test_scaled (Expected: Mean ≈ 0.0, Std ≈ 1.0, not exact due to holdout nature) ===")
print(test_verif.round(4))

print("\n=== FIRST 5 ROWS OF X_train_scaled ===")
print(X_train_scaled.head())


=== SCALING VERIFICATION: X_train_scaled (Expected: Mean ≈ 0.0, Std ≈ 1.0) ===
                     Mean       Std
eye_distance          0.0  1.000209
mouth_width           0.0  1.000209
face_width            0.0  1.000209
face_height           0.0  1.000209
face_aspect_ratio    -0.0  1.000209
eye_to_face_ratio     0.0  1.000209
mouth_to_face_ratio  -0.0  1.000209

=== SCALING VERIFICATION: X_test_scaled (Expected: Mean ≈ 0.0, Std ≈ 1.0, not exact due to holdout nature) ===
                       Mean     Std
eye_distance        -0.0134  0.9637
mouth_width         -0.0308  0.9703
face_width           0.0370  1.0046
face_height         -0.0633  0.9698
face_aspect_ratio   -0.0719  0.9758
eye_to_face_ratio   -0.0629  0.9782
mouth_to_face_ratio -0.0579  0.9765

=== FIRST 5 ROWS OF X_train_scaled ===
      gender_0  gender_1  race_0  race_1  race_2  race_3  race_4  \
1569         0         1       1       0       0       0       0   
2228         1         0       0       0       0       1 

In [6]:
# Define output file dictionary for unscaled and scaled datasets
output_files = {
    "X_train.csv": X_train,
    "X_test.csv": X_test,
    "y_train.csv": y_train,
    "y_test.csv": y_test,
    "X_train_scaled.csv": X_train_scaled,
    "X_test_scaled.csv": X_test_scaled
}

output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

# Export all datasets to CSV without index
for filename, df_to_save in output_files.items():
    filepath = os.path.join(output_dir, filename)
    df_to_save.to_csv(filepath, index=False)

print("=== DATASET EXPORT CONFIRMATION ===")
print("Successfully saved partitioned and scaled datasets to data/processed/:")
for filename, df_to_save in output_files.items():
    print(f"  - {filename:<20} Shape: {df_to_save.shape}")


=== DATASET EXPORT CONFIRMATION ===
Successfully saved partitioned and scaled datasets to data/processed/:
  - X_train.csv          Shape: (2398, 14)
  - X_test.csv           Shape: (600, 14)
  - y_train.csv          Shape: (2398,)
  - y_test.csv           Shape: (600,)
  - X_train_scaled.csv   Shape: (2398, 14)
  - X_test_scaled.csv    Shape: (600, 14)
